# 🦟 Evolução dos Casos de Dengue no Brasil
## Projeto G2 — Análise e Visualização de Dados com Python
---
**Como usar este notebook:**
1. Abra o Google Colab: https://colab.research.google.com
2. Clique em `Arquivo → Fazer upload de notebook` e selecione este arquivo
3. Execute a célula de upload e selecione o arquivo `simulacao_dengue_brasil.csv`
4. Execute todas as demais células em ordem (Ctrl+F9)
---

## 1. Introdução

A dengue é uma das principais doenças transmitidas por mosquitos no Brasil. Causada pelo vírus dengue e transmitida pelo *Aedes aegypti*, a doença está ligada a condições climáticas como chuva e temperatura elevada.

**Perguntas que este notebook responde:**
- Quais estados têm mais casos?
- Quais meses são mais críticos?
- Os casos cresceram ao longo dos anos?
- Existe relação entre chuva e dengue?
- Quais municípios têm maior incidência?

## 2. Instalação de Bibliotecas e Imports

In [ ]:
# Instala bibliotecas extras que o Colab não tem por padrão
!pip install plotly sqlalchemy --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11})

CORES = {'Norte':'#3182ce','Nordeste':'#dd6b20',
         'Centro-Oeste':'#38a169','Sudeste':'#e53e3e','Sul':'#805ad5'}
MESES = ['Jan','Fev','Mar','Abr','Mai','Jun',
         'Jul','Ago','Set','Out','Nov','Dez']

print(' Tudo importado com sucesso!')

## 3. Upload e Leitura do CSV

In [ ]:
# Esta célula abre uma janela para você escolher o arquivo CSV
from google.colab import files

print(' Clique em "Escolher arquivos" e selecione o arquivo:')
print('   simulacao_dengue_brasil.csv')
print()

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

df_raw = pd.read_csv(nome_arquivo)
print(f'\n Arquivo carregado: {nome_arquivo}')
print(f' {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas')
df_raw.head()

## 4. Limpeza e Preparação dos Dados

In [ ]:
print('=== INFORMAÇÕES DO DATASET ===')
df_raw.info()
print('\n=== VALORES NULOS ===')
print(df_raw.isnull().sum())
print('\n=== ESTATÍSTICAS DESCRITIVAS ===')
display(df_raw.describe().round(2))

In [ ]:
df = df_raw.copy()

# Converter tipos
df['data'] = pd.to_datetime(df['data'])
for c in ['ano','mes','populacao','casos_dengue','internacoes','obitos']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
for c in ['chuva_mm','temperatura_media','incidencia_100k']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Padronizar textos
df['regiao']       = df['regiao'].str.strip().str.title()
df['uf']           = df['uf'].str.strip().str.upper()
df['municipio']    = df['municipio'].str.strip().str.title()
df['nivel_alerta'] = df['nivel_alerta'].str.strip().str.title()

# Remover duplicatas
antes = len(df)
df = df.drop_duplicates()
print(f'Duplicatas removidas: {antes - len(df)}')
print(f' Dataset limpo: {df.shape[0]:,} linhas | Nulos: {df.isnull().sum().sum()}')

## 5. Engenharia de Atributos (Novas Colunas)

In [ ]:
# Nome do mês
df['nome_mes'] = df['mes'].apply(lambda x: MESES[x-1])

# Estação do ano (hemisfério Sul)
def estacao(m):
    if m in [12,1,2]:  return 'Verão'
    if m in [3,4,5]:   return 'Outono'
    if m in [6,7,8]:   return 'Inverno'
    return 'Primavera'
df['estacao'] = df['mes'].apply(estacao)

# Taxas derivadas
df['taxa_letalidade'] = np.where(df['casos_dengue']>0,
    df['obitos']/df['casos_dengue']*100, 0).round(4)
df['taxa_internacao'] = np.where(df['casos_dengue']>0,
    df['internacoes']/df['casos_dengue']*100, 0).round(2)

print(' Novas colunas criadas:')
df[['nome_mes','estacao','taxa_letalidade','taxa_internacao']].head(3)

## 6. KPIs — Indicadores Principais

In [ ]:
total_casos  = df['casos_dengue'].sum()
total_ob     = df['obitos'].sum()
total_int    = df['internacoes'].sum()
media_men    = df.groupby(['ano','mes'])['casos_dengue'].sum().mean()
inc_media    = df['incidencia_100k'].mean()
uf_top       = df.groupby('uf')['casos_dengue'].sum().idxmax()
uf_top_v     = df.groupby('uf')['casos_dengue'].sum().max()
mun_top      = df.groupby('municipio')['incidencia_100k'].mean().idxmax()
mun_inc      = df.groupby('municipio')['incidencia_100k'].mean().max()
ano_pico     = df.groupby('ano')['casos_dengue'].sum().idxmax()
ano_pico_v   = df.groupby('ano')['casos_dengue'].sum().max()

print('=' * 55)
print('         PAINEL DE KPIs - DENGUE NO BRASIL')
print('=' * 55)
print(f'   Total de Casos          : {total_casos:>12,.0f}')
print(f'   Total de Óbitos         : {total_ob:>12,.0f}')
print(f'   Total de Internações    : {total_int:>12,.0f}')
print(f'   Média Mensal de Casos   : {media_men:>12,.0f}')
print(f'   Incidência Média /100k  : {inc_media:>12.1f}')
print(f'    Estado Mais Afetado    : {uf_top} ({uf_top_v:,.0f} casos)')
print(f'   Município Crítico       : {mun_top} ({mun_inc:.1f}/100k)')
print(f'   Ano de Pico             : {ano_pico} ({ano_pico_v:,.0f} casos)')
print('=' * 55)

## 7. Gráficos

### 7.1 Evolução Temporal dos Casos

In [ ]:
por_ano = df.groupby('ano').agg(
    casos=('casos_dengue','sum'),
    obitos=('obitos','sum'),
    internacoes=('internacoes','sum')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linha de casos
axes[0].plot(por_ano['ano'], por_ano['casos'],
             marker='o', lw=2.5, color='#e53e3e', ms=8)
axes[0].fill_between(por_ano['ano'], por_ano['casos'],
                      alpha=0.15, color='#e53e3e')
axes[0].set_title('Evolução Anual dos Casos')
axes[0].set_xlabel('Ano'); axes[0].set_ylabel('Total de Casos')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for x,y in zip(por_ano['ano'], por_ano['casos']):
    axes[0].annotate(f'{y:,.0f}', (x,y),
                     xytext=(0,8), textcoords='offset points',
                     ha='center', fontsize=8)

# Barras óbitos/internações
x = np.arange(len(por_ano)); w = 0.38
axes[1].bar(x-w/2, por_ano['obitos'],     w,
            label='Óbitos',      color='#8c1a11')
axes[1].bar(x+w/2, por_ano['internacoes'],w,
            label='Internações', color='#ed8936')
axes[1].set_title('Óbitos e Internações por Ano')
axes[1].set_xticks(x)
axes[1].set_xticklabels(por_ano['ano'], rotation=45)
axes[1].legend()

plt.suptitle('Série Histórica - Dengue no Brasil (2015-2024)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**📝 Interpretação:** Observe os anos com picos de casos — geralmente ligados à reintrodução de sorotipos virais ou condições climáticas favoráveis.

### 7.2 Comparação entre Regiões

In [ ]:
por_reg = df.groupby('regiao')['casos_dengue'].sum()\
            .sort_values(ascending=False).reset_index()
por_reg['pct'] = (por_reg['casos_dengue']/por_reg['casos_dengue'].sum()*100).round(1)
cores_lista = [CORES.get(r,'#aaa') for r in por_reg['regiao']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(por_reg['regiao'], por_reg['casos_dengue'], color=cores_lista)
axes[0].set_title('Total de Casos por Região')
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for i,(v,p) in enumerate(zip(por_reg['casos_dengue'], por_reg['pct'])):
    axes[0].text(v+100, i, f' {v:,.0f} ({p}%)', va='center', fontsize=9)

axes[1].pie(por_reg['casos_dengue'], labels=por_reg['regiao'],
            autopct='%1.1f%%', colors=cores_lista, startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Distribuição % por Região')

plt.suptitle('Casos de Dengue por Região', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Evolução por região - Plotly interativo
por_reg_ano = df.groupby(['ano','regiao'])['casos_dengue'].sum().reset_index()
fig = px.line(por_reg_ano, x='ano', y='casos_dengue', color='regiao',
              markers=True, color_discrete_map=CORES,
              title='Evolução por Região (2015-2024)',
              labels={'casos_dengue':'Casos','ano':'Ano','regiao':'Região'})
fig.update_layout(hovermode='x unified')
fig.show()

### 7.3 Comparação entre Estados (UF)

In [ ]:
por_uf = df.groupby(['uf','regiao'])['casos_dengue'].sum()\
           .reset_index().sort_values('casos_dengue', ascending=False)
cores_uf = [CORES.get(r,'#aaa') for r in por_uf['regiao']]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(por_uf['uf'], por_uf['casos_dengue'], color=cores_uf, edgecolor='white')
ax.set_title('Casos de Dengue por Estado (UF)', fontsize=13, fontweight='bold')
ax.set_xlabel('Estado'); ax.set_ylabel('Total de Casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v,label=k) for k,v in CORES.items()],
          title='Região', loc='upper right')
plt.tight_layout()
plt.show()

### 7.4 Ranking de Municípios

In [ ]:
rank = df.groupby(['municipio','uf','regiao'])\
         .agg(casos=('casos_dengue','sum'),
              incidencia=('incidencia_100k','mean'),
              obitos=('obitos','sum'))\
         .reset_index()\
         .sort_values('incidencia', ascending=False)\
         .head(15)

cores_m = [CORES.get(r,'#aaa') for r in rank['regiao']]
labels  = rank['municipio'] + '/' + rank['uf']

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(labels[::-1], rank['incidencia'][::-1], color=cores_m[::-1])
ax.set_title('Top 15 Municípios - Incidência Média por 100k hab.',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Incidência Média / 100 mil habitantes')
for i,v in enumerate(rank['incidencia'][::-1]):
    ax.text(v+0.3, i, f'{v:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\n Tabela - Top 15 Municípios')
display(rank.reset_index(drop=True).style.format(
    {'casos':'{:,.0f}','incidencia':'{:.1f}','obitos':'{:,.0f}'}
).background_gradient(subset=['incidencia'], cmap='OrRd'))

### 7.5 Heatmap de Sazonalidade

In [ ]:
pivot = df.groupby(['ano','mes'])['casos_dengue'].sum()\
          .unstack('mes').fillna(0)
pivot.columns = [MESES[c-1] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Casos'}, ax=ax)
ax.set_title('Heatmap de Sazonalidade - Casos por Mês e Ano',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mês'); ax.set_ylabel('Ano')
plt.tight_layout()
plt.show()

# Média mensal
sazon = df.groupby('mes')['casos_dengue'].mean().reset_index()
sazon['nome_mes'] = sazon['mes'].apply(lambda x: MESES[x-1])
mes_pico = sazon.loc[sazon['casos_dengue'].idxmax(), 'nome_mes']

cores_b = ['#e53e3e' if v==sazon['casos_dengue'].max()
           else '#aec7e8' for v in sazon['casos_dengue']]
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(sazon['nome_mes'], sazon['casos_dengue'],
       color=cores_b, edgecolor='white')
ax.set_title('Média Histórica Mensal de Casos', fontsize=13, fontweight='bold')
ax.set_ylabel('Média de Casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
for i,v in enumerate(sazon['casos_dengue']):
    ax.text(i, v+5, f'{v:,.0f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print(f' Mês historicamente mais crítico: {mes_pico}')

**📝 Interpretação:** Células mais escuras indicam os períodos mais críticos. Os meses de jan–abr concentram a maior carga, coincidindo com o verão no hemisfério Sul.

### 7.6 Correlação Chuva × Dengue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for reg, grp in df.groupby('regiao'):
    axes[0].scatter(grp['chuva_mm'], grp['casos_dengue'],
                    alpha=0.4, s=18, label=reg,
                    color=CORES.get(reg,'#aaa'))
axes[0].set_title('Dispersão: Chuva × Casos de Dengue')
axes[0].set_xlabel('Chuva (mm)')
axes[0].set_ylabel('Casos de Dengue')
axes[0].legend(title='Região', markerscale=2)

med = df.groupby('mes').agg(
    chuva=('chuva_mm','mean'), casos=('casos_dengue','mean')).reset_index()
ax2 = axes[1].twinx()
axes[1].bar(med['mes'], med['chuva'],
            color='#4299e1', alpha=0.6, label='Chuva (mm)')
ax2.plot(med['mes'], med['casos'],
         color='#e53e3e', marker='o', lw=2.5, label='Casos médios')
axes[1].set_title('Chuva × Casos - Médias Mensais')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Chuva (mm)', color='#4299e1')
ax2.set_ylabel('Casos médios', color='#e53e3e')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(MESES)
plt.tight_layout()
plt.show()

r = df[['chuva_mm','casos_dengue']].corr().iloc[0,1]
forca = 'FORTE' if abs(r)>.6 else 'MODERADA' if abs(r)>.3 else 'FRACA'
print(f'\n Correlação de Pearson (chuva × casos): r = {r:.4f}    {forca}')

In [ ]:
# Matriz de correlação
cols = ['chuva_mm','temperatura_media','casos_dengue',
        'internacoes','obitos','incidencia_100k']
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(df[cols].corr(), dtype=bool))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1,
            mask=mask, linewidths=0.5, square=True, ax=ax)
ax.set_title('Matriz de Correlação', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.7 Tabela Dinâmica — Casos por UF e Ano

In [ ]:
tabela = pd.pivot_table(df, index='uf', columns='ano',
                         values='casos_dengue', aggfunc='sum',
                         fill_value=0, margins=True, margins_name='TOTAL')
tabela.columns = [str(c) for c in tabela.columns]

print(' Casos de Dengue por Estado e Ano')
display(tabela.style
        .format('{:,.0f}')
        .background_gradient(cmap='Reds',
                              subset=pd.IndexSlice[:, tabela.columns[:-1]])
        .set_caption('Tabela Dinâmica - Casos por UF × Ano'))

## 8. Persistência em SQLite

In [ ]:
engine = create_engine('sqlite:///dengue_brasil.sqlite', echo=False)
df.to_sql('dengue', engine, if_exists='replace', index=False)
print(' Dados salvos em dengue_brasil.sqlite')

consultas = {
    'Top 5 estados por casos': '''
        SELECT uf, SUM(casos_dengue) AS total
        FROM dengue GROUP BY uf
        ORDER BY total DESC LIMIT 5''',
    'Casos e óbitos por ano': '''
        SELECT ano, SUM(casos_dengue) AS casos,
               SUM(obitos) AS obitos,
               ROUND(AVG(incidencia_100k),2) AS incidencia
        FROM dengue GROUP BY ano ORDER BY ano''',
    'Meses mais críticos': '''
        SELECT mes, SUM(casos_dengue) AS total
        FROM dengue GROUP BY mes
        ORDER BY total DESC LIMIT 6'''
}

with engine.connect() as conn:
    for titulo, sql in consultas.items():
        print(f'\n=== {titulo} ===')
        display(pd.read_sql_query(text(sql), conn))

## 9. Interpretação dos Resultados

In [ ]:
cresc      = df.groupby('ano')['casos_dengue'].sum().pct_change().mean()*100
mes_crit   = MESES[df.groupby('mes')['casos_dengue'].sum().idxmax()-1]
reg_lider  = df.groupby('regiao')['casos_dengue'].sum().idxmax()
r          = df[['chuva_mm','casos_dengue']].corr().iloc[0,1]
forca      = 'forte' if abs(r)>.6 else 'moderada' if abs(r)>.3 else 'fraca'

print('=' * 60)
print('           INTERPRETAÇÃO DOS RESULTADOS')
print('=' * 60)
print(f'''
 EVOLUÇÃO TEMPORAL
   {total_casos:,.0f} casos entre 2015 e 2024.
   Crescimento médio anual: {cresc:+.1f}%.
   Ano de pico: {ano_pico} ({ano_pico_v:,.0f} casos).

 SAZONALIDADE
   Mês mais crítico: {mes_crit}.
   Período de risco: janeiro-abril (verão/outono).

 DISTRIBUIÇÃO REGIONAL
   Região com mais casos: {reg_lider}.
   Estado mais afetado: {uf_top} ({uf_top_v:,.0f} casos).

 CORRELAÇÃO CHUVA × DENGUE
   r = {r:.3f} - correlação {forca}.
   Defasagem típica de 2-4 semanas (ciclo do mosquito).

 MUNICÍPIO CRÍTICO
   {mun_top} - {mun_inc:.1f} casos/100k hab.
''')
print('=' * 60)

## 10. Conclusão

| Pergunta | Resposta |
|---|---|
| Estados com mais casos? | Norte e Nordeste lideram |
| Períodos críticos? | Janeiro a abril |
| Crescimento ao longo dos anos? | Tendência de alta |
| Relação chuva × dengue? | Correlação positiva confirmada |
| Municípios com maior incidência? | Ver ranking — seção 7.4 |
| Regiões mais vulneráveis? | Norte e Nordeste |
| Quando agir preventivamente? | Outubro a dezembro |

### Recomendações
1. Campanhas de eliminação de focos em **outubro–dezembro**
2. Reforço hospitalar em **janeiro–abril**
3. Vigilância intensiva nos municípios críticos
4. Alertas climáticos integrados com previsão de chuvas

---
> *Projeto G2 · Dados simulados para fins didáticos.*